# 13 — Model Export and Inference

## Objective
Explain the adapter export path and show the required commands without pretending external conversion tools are installed.

## Background
PEFT (Parameter-Efficient Fine-Tuning) adapters store weight updates rather than a complete model. Ollama usually consumes GGUF (GGML Unified Format) model files or supported model assets. Conversion and quantization require external tools and must be validated on the actual target architecture.

### Locate saved artifacts

**Procedure.** Resolve the directory containing the trained SFT adapter and export destination.

**Reading the result.** Printed paths identify input and output locations for the later optional export.

In [1]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


### Check export utilities

**Procedure.** Check whether Python, Git, Ollama and quantization commands are installed.

**Reading the result.** A missing command means its corresponding export or serving stage may be unavailable.

In [2]:
# Purpose: Check whether Python, Git, Ollama and quantization commands are installed.
# Result: A missing command means its corresponding export or serving stage may be
#   unavailable.
import shutil
for command in ['ollama','llama-quantize','python','git']:
    print(command, '=>', shutil.which(command) or 'not installed')
for name in ['sft_adapter','dpo_adapter','grpo_adapter']:
    print(name, (RUNS/name/'adapter_config.json').exists())

ollama => C:\Users\weiju\AppData\Local\Programs\Ollama\ollama.EXE
llama-quantize => not installed
python => C:\Python314\python.EXE
git => C:\Program Files\Git\cmd\git.EXE
sft_adapter True
dpo_adapter True
grpo_adapter True


### Export a full model checkpoint when enabled

**Procedure.** Load the base checkpoint and SFT adapter, merge the adapter weights, and save the merged model and tokenizer.

**Reading the result.** The merged directory is an input for a separate GGUF conversion step; this cell does not by itself create an Ollama model.

In [3]:
import json
import os

if os.getenv("RUN_EXPORT", "0") == "1":
    import hashlib
    import time
    import torch
    import transformers
    import peft
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftConfig, PeftModel

    adapter_dir = RUNS / "sft_adapter"
    manifest_path = adapter_dir / "run_manifest.json"
    adapter_path = adapter_dir / "adapter_model.safetensors"
    if not all(path.exists() for path in (manifest_path, adapter_path,
                                           adapter_dir / "adapter_config.json")):
        raise FileNotFoundError("Verified SFT adapter files are required for merging.")
    provenance = json.loads(manifest_path.read_text(encoding="utf-8"))
    base_model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
    revision = provenance["model_revision"]
    config = PeftConfig.from_pretrained(str(adapter_dir))
    if provenance["model_id"] != base_model_id or config.base_model_name_or_path != base_model_id:
        raise ValueError("Adapter and requested base model identities differ.")
    if not revision or len(revision) != 40:
        raise ValueError("SFT run manifest lacks a pinned base revision.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Merge in float32 to avoid rounding each LoRA delta into bfloat16 base weights.
    dtype = torch.float32
    started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, revision=revision,
                                                local_files_only=True)
    base = AutoModelForCausalLM.from_pretrained(base_model_id, revision=revision,
             local_files_only=True, dtype=dtype).to(device)
    if getattr(base.config, "_commit_hash", None) != revision:
        raise ValueError("Loaded base checkpoint differs from training lineage.")
    tuned = PeftModel.from_pretrained(base, str(adapter_dir)).eval()
    # Fixed synthetic prompt checks the merge numerically without using validation/test labels.
    prompt = tokenizer.apply_chat_template([
        {"role": "system", "content": "Answer with exactly one label: yes, no, or maybe."},
        {"role": "user", "content": "Question: Is this an export consistency check?"},
    ], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    with torch.inference_mode():
        adapter_logits = tuned(**inputs).logits[:, -1, :].float().cpu()
    merged = tuned.merge_and_unload().eval()
    with torch.inference_mode():
        merged_logits = merged(**inputs).logits[:, -1, :].float().cpu()
    merge_max_error = (adapter_logits - merged_logits).abs().max().item()
    if merge_max_error > 0.1 or adapter_logits.argmax() != merged_logits.argmax():
        raise RuntimeError(f"Merged model disagrees with adapter-loaded model: {merge_max_error:.4g}")

    output_dir = RUNS / "merged_sft"
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError("Existing merged export found; inspect before replacing it.")
    output_dir.mkdir(exist_ok=True)
    merged.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    del merged, tuned, base
    if device == "cuda":
        torch.cuda.empty_cache()
    reloaded = AutoModelForCausalLM.from_pretrained(
        output_dir, local_files_only=True, dtype=dtype).to(device).eval()
    reloaded_tokenizer = AutoTokenizer.from_pretrained(output_dir, local_files_only=True)
    if reloaded_tokenizer.apply_chat_template([
        {"role": "system", "content": "Answer with exactly one label: yes, no, or maybe."},
        {"role": "user", "content": "Question: Is this an export consistency check?"},
    ], tokenize=False, add_generation_prompt=True) != prompt:
        raise RuntimeError("Reloaded tokenizer changed chat formatting.")
    with torch.inference_mode():
        reloaded_logits = reloaded(**inputs).logits[:, -1, :].float().cpu()
    reload_max_error = (merged_logits - reloaded_logits).abs().max().item()
    if reload_max_error > 0.01 or merged_logits.argmax() != reloaded_logits.argmax():
        raise RuntimeError(f"Reloaded checkpoint differs from merged model: {reload_max_error:.4g}")

    def sha256(path):
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    weights = sorted(output_dir.glob("*.safetensors"))
    if not weights:
        raise FileNotFoundError("Merged model contains no safetensors weights.")
    export = {
        "base_model": base_model_id, "base_revision": revision,
        "source_adapter": str(adapter_dir), "source_adapter_sha256": sha256(adapter_path),
        "source_dataset_fingerprint": provenance["dataset_fingerprint"],
        "export_type": "merged Hugging Face checkpoint; not GGUF or Ollama",
        "dtype": str(dtype), "torch_version": torch.__version__,
        "transformers_version": transformers.__version__, "peft_version": peft.__version__,
        "weight_files_sha256": {p.name: sha256(p) for p in weights},
        "merge_max_logit_error": merge_max_error,
        "reload_max_logit_error": reload_max_error,
        "probe_next_token_id": int(reloaded_logits.argmax()),
        "elapsed_seconds": round(time.perf_counter() - started, 3),
    }
    (output_dir / "export_manifest.json").write_text(
        json.dumps(export, indent=2) + "\n", encoding="utf-8")
    print("Exported and reloaded merged Hugging Face model:", output_dir)
    print("Maximum last-token logit difference, adapter versus merge:", merge_max_error)
    print("Maximum last-token logit difference, merge versus reload:", reload_max_error)
    print("GGUF conversion, quantization, and Ollama creation were not performed.")
else:
    print("Merge disabled. Set RUN_EXPORT=1 only with an existing verified adapter.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Exported and reloaded merged Hugging Face model: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs\merged_sft
Maximum last-token logit difference, adapter versus merge: 5.173683166503906e-05
Maximum last-token logit difference, merge versus reload: 0.0
GGUF conversion, quantization, and Ollama creation were not performed.


### Results

The SFT adapter merged into a float32 Hugging Face checkpoint using the pinned Qwen base. On one synthetic probe, adapter-versus-merge last-token logits differed by at most `5.17e-05`; reloading the merged checkpoint reproduced its logits exactly. Ollama 0.33.3 was present, but `llama-quantize` was unavailable.

### Interpretation

The local merge, source hashes, and checkpoint reload succeeded for the tested probe.

### Limitations

One probe does not establish broad answer equivalence. No GGUF conversion, Ollama model creation, service inference, or deployment validation occurred.

### Key Observations

Only a merged Hugging Face SFT checkpoint was produced; the DPO/GRPO adapters were not exported here.


### Conversion and integration
After an actual merge, use the current `llama.cpp` conversion script compatible with your model, validate the GGUF conversion, quantize it, then create an Ollama model using a checked `Modelfile`. The deployment tree also supports direct Hugging Face/PEFT inference without GGUF conversion.

## Key observations and technical discussion
Only mark conversion and smoke tests complete after tools exist and the generated model returns a response. Export does not establish model correctness.

## Q&A
**Question:** Is saving a low-rank adapter equivalent to exporting a complete inference checkpoint?

**Answer:** No. The base checkpoint is still needed until adapters are merged or loaded alongside it.

### Reproducibility note
Each printed example belongs to one of three categories: an illustrative unit check, a source-backed inspection, or an opt-in model experiment. Only real executed experiments with source/model identifiers and saved outputs can support research findings.